<a href="https://colab.research.google.com/github/SaraMo45/Next.js/blob/main/Equipo_6_Segunda_entrega.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# ---------------------------------------------------------------
#  MÉTODO DE STANDING
# ---------------------------------------------------------------
def ratio_vogel(pp, pR):
    """q/qmax(EF=1) con la ecuación de Standing (forma de Vogel) evaluada en p'wf."""
    r = pp / pR
    return 1 - 0.2 * r - 0.8 * r**2

def p_wf_prima(pwf, pR, EF):
    """Presión de fondo fluyendo con daño: p'wf = pR - (pR - pwf)*EF"""
    return pR - (pR - pwf) * EF

def pwf_minima(pR, EF):
    """Límite de validez de Standing (p'wf >= 0): pwf >= pR*(1 - 1/EF). Solo restringe si EF > 1."""
    return max(0.0, pR * (1 - 1 / EF))

def ratio_standing(pwf, pR, EF):
    """
    q/qmax(EF=1) para un valor de pwf con la ecuación de Standing.
    Devuelve NaN donde p'wf < 0 (fuera del rango de validez del método).
    """
    pwf = np.asarray(pwf, dtype=float)
    pp = p_wf_prima(pwf, pR, EF)
    return np.where(pp >= -1e-9, ratio_vogel(pp, pR), np.nan)

def standing_qmax_EF1(q_test, pwf_test, pR, EF):
    """Paso 1: gasto máximo sin daño (EF=1) a partir de la prueba."""
    r = float(ratio_standing(pwf_test, pR, EF))
    if not np.isfinite(r) or r <= 0:
        raise ValueError("La prueba queda fuera del rango de validez de Standing (p'wf < 0): "
                         "use una pwf de prueba mayor o una EF menor.")
    return q_test / r

def standing_qmax_EF_aprox(qmax1, EF):
    """Expresiones de las notas para el gasto máximo según el valor de EF."""
    if EF < 1:
        return qmax1 * (1.8 * EF - 0.8 * EF**2)
    if EF > 1:
        return qmax1 * (0.624 + 0.376 * EF)
    return qmax1

def standing_qmax_EF(pR, qmax1, EF):
    """
    Paso 2: gasto máximo a la EF deseada (pwf = 0).
    - EF <= 1: ecuación de Standing con p'wf = pR*(1 - EF).
    - EF  > 1: p'wf resulta negativa en pwf = 0; se usa la expresión de las notas
               qmax(EF>1) = qmax(EF=1)*(0.624 + 0.376*EF).
    """
    if EF <= 1:
        return float(qmax1 * ratio_vogel(pR * (1 - EF), pR))
    return float(standing_qmax_EF_aprox(qmax1, EF))

def standing_curva(pR, qmax1, EF, n=60):
    """Curva IPR dentro del rango de validez (pwf desde pwf_minima hasta pR)."""
    pwf = np.linspace(pwf_minima(pR, EF), pR, n)
    q = qmax1 * ratio_standing(pwf, pR, EF)
    return pwf, q

# ---------------------------------------------------------------
#  MÉTODO DE JONES, BLOUNT Y GLAZE (JBG)
# ---------------------------------------------------------------
def jbg_ajuste(q, pwf, pws):
    """(pws - pwf)/q = a*q + b  -> regresión lineal por mínimos cuadrados."""
    q = np.asarray(q, float); pwf = np.asarray(pwf, float)
    y = (pws - pwf) / q
    a, b = np.polyfit(q, y, 1)
    yhat = a * q + b
    ss_res = np.sum((y - yhat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 1.0
    return a, b, r2, y

def jbg_qmax(a, b, pws):
    """Raíz positiva de a*q^2 + b*q - pws = 0 (pwf = 0)."""
    if abs(a) < 1e-14:
        return pws / b
    disc = b**2 + 4 * a * pws
    return (-b + np.sqrt(disc)) / (2 * a)

def jbg_ipr(a, b, pws, n=60):
    qmax = jbg_qmax(a, b, pws)
    q = np.linspace(0, qmax, n)
    return q, pws - a * q**2 - b * q

def jbg_diagnostico(a, b, bp):
    ratio = bp / b if b > 0 else np.inf
    dano = b > 0.05
    turb = ratio > 2
    txt_d = "HAY daño (b > 0.05)" if dano else "sin daño (b <= 0.05)"
    txt_t = "turbulencia significativa (b'/b > 2)" if turb else "turbulencia nula o pequeña (b'/b <= 2)"
    lineas = [f"b = {b:.5f}  ->  {txt_d}",
              f"b'/b = {ratio:.3f}  ->  {txt_t}"]
    if not dano and not turb:
        rec = "La terminación del pozo es buena."
    elif not dano and turb:
        rec = ("No se recomienda estimular. La baja productividad se debe a un área insuficiente "
               "del intervalo disparado: se recomienda incrementar el área disparada.")
    elif dano and not turb:
        rec = "Se recomienda un trabajo de estimulación."
    else:
        rec = ("Hay daño y turbulencia importantes (caso no cubierto explícitamente por los criterios): "
               "se sugiere estimular y evaluar también incrementar el área disparada.")
    if a < 0:
        rec += "  AVISO: pendiente 'a' negativa (turbulencia no física); revisa los datos de prueba."
    if b <= 0:
        rec += "  AVISO: b <= 0 (no físico); el ajuste no es confiable, revisa los datos de prueba."
    return lineas, rec

def _fmt_table(df):
    return df.round(2).to_string(index=False, na_rep="—")

def correr_standing(pR, q_test, pwf_test, EF_test, EF_extra, n_pts=25, mostrar=True):
    """Ejecuta Standing completo: qmax(EF=1), qmax(EF), tabla IPR y gráfica."""
    qmax1 = standing_qmax_EF1(q_test, pwf_test, pR, EF_test)
    EFs = sorted(set([round(EF_test, 4), 1.0] + [round(e, 4) for e in EF_extra]))

    resumen = pd.DataFrame({
        "EF": EFs,
        "qo máx [bpd]": [standing_qmax_EF(pR, qmax1, e) for e in EFs],
        "Cálculo": ["Ec. Standing" if e <= 1 else "Expresión notas (EF>1)" for e in EFs],
        "pwf mín. válida [lb/pg²]": [pwf_minima(pR, e) for e in EFs],
    })

    pwf_g = np.linspace(0, pR, n_pts)
    tabla = pd.DataFrame({"pwf [lb/pg²]": pwf_g})
    for e in EFs:
        tabla[f"qo EF={e:g} [bpd]"] = qmax1 * ratio_standing(pwf_g, pR, e)

    fig, ax = plt.subplots(figsize=(7.5, 5))
    for e in EFs:
        pw, q = standing_curva(pR, qmax1, e)
        lw = 2.8 if e == round(EF_test, 4) else 1.6
        line, = ax.plot(q, pw, lw=lw, label=f"EF = {e:g}")
        if e > 1:
            ax.plot(standing_qmax_EF(pR, qmax1, e), 0, "o", mfc="white", color=line.get_color(), ms=7)
    ax.plot(q_test, pwf_test, "ko", ms=8, label="Prueba de producción")
    ax.set_xlabel("Gasto de aceite, qo [bpd]")
    ax.set_ylabel("Presión de fondo fluyendo, pwf [lb/pg²]")
    ax.set_title("Curvas IPR – Método de Standing")
    ax.set_xlim(left=0); ax.set_ylim(0, pR * 1.02)
    ax.legend()
    plt.tight_layout()

    if mostrar:
        print("=" * 62)
        print("RESULTADOS – MÉTODO DE STANDING")
        print("=" * 62)
        print(f"Datos: p̄R = {pR:g} | qo = {q_test:g} bpd | pwf = {pwf_test:g} | EF = {EF_test:g}")
        print(f"p'wf de la prueba = {p_wf_prima(pwf_test, pR, EF_test):.1f} lb/pg²")
        print(f"qo máx (EF=1) = {qmax1:.2f} bpd\n")
        print(_fmt_table(resumen))
        if any(e > 1 for e in EFs):
            print("\nNota: con EF>1 la ecuación de Standing solo es válida si p'wf >= 0, es decir "
                  "pwf >= p̄R(1-1/EF); fuera de ese rango la tabla muestra «—» y la curva no se dibuja."
                  "\nEl qo máx con EF>1 (círculo hueco) se calcula con la expresión de las notas: "
                  "qo máx(EF=1)*(0.624+0.376*EF).")
        print("\nTabla de la curva IPR:")
        print(_fmt_table(tabla))
        plt.show()
    return qmax1, resumen, tabla

def correr_jbg(pws, q, pwf, mostrar=True):
    a, b, r2, y = jbg_ajuste(q, pwf, pws)
    qmax = jbg_qmax(a, b, pws)
    bp = b + a * qmax
    lineas, rec = jbg_diagnostico(a, b, bp)

    qq, pp = jbg_ipr(a, b, pws)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.6))
    qline = np.linspace(0, max(q) * 1.15, 50)
    axs[0].plot(q, y, "ko", label="Datos de prueba")
    axs[0].plot(qline, a * qline + b, "r-", label=f"Ajuste: y = {a:.3e}·q + {b:.4f}")
    axs[0].plot(0, b, "rs", label=f"b = {b:.4f}")
    axs[0].set_xlim(left=0)
    axs[0].set_xlabel("Gasto, q [bpd]"); axs[0].set_ylabel("(pws − pwf)/q  [lb/pg²/bpd]")
    axs[0].set_title("Gráfica de diagnóstico JBG"); axs[0].legend()
    axs[1].plot(qq, pp, "b-", lw=2, label="IPR (JBG)")
    axs[1].plot(q, pwf, "ko", label="Datos de prueba")
    axs[1].plot(qmax, 0, "r^", ms=9, label=f"qo máx = {qmax:.0f} bpd")
    axs[1].set_xlim(left=0); axs[1].set_ylim(0, pws * 1.02)
    axs[1].set_xlabel("Gasto de aceite, qo [bpd]"); axs[1].set_ylabel("pwf [lb/pg²]")
    axs[1].set_title("Curva IPR – Método de JBG"); axs[1].legend()
    plt.tight_layout()

    if mostrar:
        print("=" * 62)
        print("RESULTADOS – MÉTODO DE JONES, BLOUNT Y GLAZE")
        print("=" * 62)
        print(f"pws = {pws:g} lb/pg² | puntos de prueba = {len(q)}")
        print(f"a (turbulencia) = {a:.6e}   [lb/pg²/bpd²]")
        print(f"b (daño)        = {b:.6f}   [lb/pg²/bpd]")
        print(f"R² del ajuste   = {r2:.4f}")
        print(f"qo máx (pwf=0)  = {qmax:.2f} bpd")
        print(f"b' = b + a·qo máx = {bp:.6f}")
        print("\nDIAGNÓSTICO:")
        for l in lineas: print("  •", l)
        print("RECOMENDACIÓN:", rec)
        plt.show()
    return dict(a=a, b=b, r2=r2, qmax=qmax, bp=bp, rec=rec)

# ===============================================================
#  ENTRADA DE DATOS Y MENÚ
# ===============================================================
def pedir(texto, tipo=float):
    """Pide un dato por teclado y repite la pregunta si el valor no es válido."""
    while True:
        try:
            return tipo(input(texto).strip().replace(",", ""))
        except ValueError:
            print("  (!) Valor no válido, escríbelo de nuevo.")

def pedir_standing():
    print("\n=== MÉTODO DE STANDING ===")
    while True:
        pR     = pedir("Presión media del yacimiento pR [lb/pg2]: ")
        q_test = pedir("Gasto de la prueba qo [bpd]: ")
        pwf    = pedir("Presión de fondo fluyendo de la prueba pwf [lb/pg2]: ")
        EF     = pedir("Eficiencia de flujo actual EF: ")
        if pR <= 0 or q_test <= 0 or EF <= 0:
            print("(!) pR, qo y EF deben ser positivos. Vuelve a capturar los datos.\n"); continue
        if not (0 <= pwf < pR):
            print("(!) Debe cumplirse 0 <= pwf < pR. Vuelve a capturar los datos.\n"); continue
        break
    txt = input("Otras EF a comparar, separadas por comas (Enter para ninguna): ").strip()
    extra = [float(x) for x in txt.replace(";", ",").split(",") if x.strip()] if txt else []
    try:
        correr_standing(pR, q_test, pwf, EF, extra)
    except Exception as e:
        print("(!) Error:", e)

def pedir_jbg():
    print("\n=== MÉTODO DE JONES, BLOUNT Y GLAZE (JBG) ===")
    while True:
        pws = pedir("Presión estática del yacimiento pws [lb/pg2]: ")
        n   = pedir("Número de puntos de prueba (mínimo 2): ", int)
        if pws <= 0 or n < 2:
            print("(!) pws debe ser positiva y se requieren al menos 2 puntos.\n"); continue
        q, pwf = [], []
        for i in range(n):
            q.append(pedir(f"  Punto {i+1} - gasto q [bpd]: "))
            pwf.append(pedir(f"  Punto {i+1} - presión de fondo fluyendo pwf [lb/pg2]: "))
        if any(v <= 0 for v in q) or any(p < 0 or p >= pws for p in pwf):
            print("(!) Los gastos deben ser positivos y 0 <= pwf < pws. Vuelve a capturar los datos.\n"); continue
        break
    try:
        correr_jbg(pws, q, pwf)
    except Exception as e:
        print("(!) Error:", e)

while True:
    print("\n" + "=" * 62)
    print("CURVAS IPR CON DAÑO A LA FORMACIÓN")
    print("  1) Método de Standing")
    print("  2) Método de Jones, Blount y Glaze (JBG)")
    print("  3) Salir")
    op = input("Elige una opción (1, 2 o 3): ").strip()
    if op == "1":
        pedir_standing()
    elif op == "2":
        pedir_jbg()
    elif op == "3":
        print("Fin del programa.")
        break
    else:
        print("  (!) Opción no válida.")